In [5]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, clear_output
import ipywidgets as widgets

## 1. Load HDF5 File

In [6]:
# Load the HDF5 file
file_path = "/home/xinhai/Documents/automoma/output/collect/traj/summit_franka/scene_0_seed_0/7221/camera_data/episode000000.hdf5"
file = h5py.File(file_path, "r")

# Get environment information
env_info = file["env_info"]
robot_name = env_info["robot_name"][()].decode("utf-8") if isinstance(env_info["robot_name"][()], bytes) else env_info["robot_name"][()]
scene_id = env_info["scene_id"][()].decode("utf-8") if isinstance(env_info["scene_id"][()], bytes) else env_info["scene_id"][()]
object_id = env_info["object_id"][()].decode("utf-8") if isinstance(env_info["object_id"][()], bytes) else env_info["object_id"][()]
num_timesteps = env_info["num_timesteps"][()]

print(f"Robot: {robot_name}")
print(f"Scene: {scene_id}")
print(f"Object: {object_id}")
print(f"Number of timesteps: {num_timesteps}")

Robot: summit_franka
Scene: scene_0_seed_0
Object: 7221
Number of timesteps: 32


## 2. Print HDF5 File Structure

In [7]:
def print_hdf5_structure(name, obj, indent=0):
    """Recursively print the structure of an HDF5 file"""
    prefix = "  " * indent
    if isinstance(obj, h5py.Group):
        print(f"{prefix}- {name} (group)")
        for key in obj.keys():
            print_hdf5_structure(key, obj[key], indent + 1)
    elif isinstance(obj, h5py.Dataset):
        print(f"{prefix}- {name} (dataset)")
        print(f"{prefix}  shape: {obj.shape}, dtype: {obj.dtype}")

print("HDF5 File Structure:")
print("===================")
for key in file.keys():
    print_hdf5_structure(key, file[key])

HDF5 File Structure:
- env_info (group)
  - grasp_pose (dataset)
    shape: (), dtype: object
  - num_timesteps (dataset)
    shape: (), dtype: int64
  - object_id (dataset)
    shape: (), dtype: object
  - robot_name (dataset)
    shape: (), dtype: object
  - scene_id (dataset)
    shape: (), dtype: object
- obs (group)
  - depth (group)
    - ego_topdown (dataset)
      shape: (32, 240, 320), dtype: float32
    - ego_wrist (dataset)
      shape: (32, 240, 320), dtype: float32
    - fix_local (dataset)
      shape: (32, 240, 320), dtype: float32
  - eef (dataset)
    shape: (32, 7), dtype: float32
  - joint (group)
    - arm (dataset)
      shape: (32, 7), dtype: float64
    - gripper (dataset)
      shape: (32,), dtype: float64
    - mobile_base (dataset)
      shape: (32, 3), dtype: float64
  - point_cloud (dataset)
    shape: (32, 4096, 6), dtype: float64
  - rgb (group)
    - ego_topdown (dataset)
      shape: (32, 240, 320, 3), dtype: uint8
    - ego_wrist (dataset)
      shape: 

## 3. Per-Frame RGB and Depth Visualization

In [8]:
def visualize_rgb_and_depth(step):
    """Visualize RGB and depth images for a given timestep"""
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'RGB and Depth Images at Step {step}', fontsize=16)
    
    # Camera views
    cameras = ['ego_topdown', 'ego_wrist', 'fix_local']
    
    # RGB images
    rgb_group = file['obs']['rgb']
    for i, camera in enumerate(cameras):
        img_rgb = rgb_group[camera][step]  # Shape: (240, 320, 3), dtype: uint8
        axes[0, i].imshow(img_rgb)
        axes[0, i].set_title(f'RGB - {camera}')
        axes[0, i].axis('off')
    
    # Depth images
    depth_group = file['obs']['depth']
    for i, camera in enumerate(cameras):
        depth_data = depth_group[camera][step]  # Shape: (240, 320), dtype: float32
        axes[1, i].imshow(depth_data, cmap='viridis')
        axes[1, i].set_title(f'Depth - {camera}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Create slider for RGB/Depth visualization
rgb_depth_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_timesteps - 1,
    step=1,
    description='Timestep:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(widgets.interactive(visualize_rgb_and_depth, step=rgb_depth_slider))

interactive(children=(IntSlider(value=0, description='Timestep:', layout=Layout(width='500px'), max=31, style=…

## 4. Per-Frame Point Cloud Visualization

In [ ]:
def visualize_pointcloud(step, max_points=5000):
    """Visualize point cloud data
    
    Point cloud shape: (32, 4096, 6)
    Last 6 dimensions: [x, y, z, r, g, b]
    """
    
    pc_data = file['obs']['point_cloud'][step]  # Shape: (4096, 6), dtype: float64
    
    # Extract XYZ coordinates and RGB colors
    xyz = pc_data[:, :3]
    rgb = pc_data[:, 3:6]
    
    # Subsample for visualization if too many points
    if len(xyz) > max_points:
        indices = np.random.choice(len(xyz), max_points, replace=False)
        xyz = xyz[indices]
        rgb = rgb[indices]
    
    # Normalize RGB values to [0, 1] if needed
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    
    # Create 3D plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    fig.suptitle(f'Point Cloud at Step {step} ({len(xyz)} points)', fontsize=14)
    
    # Plot points with colors
    scatter = ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], 
                        c=rgb, s=1.0, alpha=0.6)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    
    plt.tight_layout()
    plt.show()

# Create slider for Point Cloud visualization
pc_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_timesteps - 1,
    step=1,
    description='Timestep:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(widgets.interactive(visualize_pointcloud, step=pc_slider))

interactive(children=(IntSlider(value=0, description='Timestep:', layout=Layout(width='500px'), max=31, style=…

## 5. Per-Frame Joint States and EEF Pose

In [ ]:
def visualize_joint_and_eef(step):
    """Visualize joint states and EEF pose data
    
    Joint data structure:
    - mobile_base: (32, 3) - [x, y, theta]
    - arm: (32, 7) - 7 joint angles
    - gripper: (32,) - gripper state
    
    EEF data structure:
    - (32, 7) - [x, y, z, qx, qy, qz, qw] (position + quaternion)
    """
    
    joint_group = file['obs']['joint']
    eef_data = file['obs']['eef'][step]  # Shape: (7,), dtype: float32
    
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    fig.suptitle(f'Joint States and EEF Pose at Step {step}', fontsize=16)
    
    # 1. Mobile Base
    ax1 = fig.add_subplot(gs[0, 0])
    mobile_base = joint_group['mobile_base'][step]  # Shape: (3,)
    ax1.bar(['X', 'Y', 'Theta'], mobile_base)
    ax1.set_title('Mobile Base Position')
    ax1.set_ylabel('Value')
    ax1.grid(True, alpha=0.3)
    
    # 2. Arm Joints
    ax2 = fig.add_subplot(gs[0, 1])
    arm_joints = joint_group['arm'][step]  # Shape: (7,)
    ax2.bar(range(len(arm_joints)), arm_joints)
    ax2.set_title('Arm Joints')
    ax2.set_xlabel('Joint Index')
    ax2.set_ylabel('Angle (rad)')
    ax2.grid(True, alpha=0.3)
    
    # 3. Gripper
    ax3 = fig.add_subplot(gs[0, 2])
    gripper = joint_group['gripper'][step]  # Shape: ()
    ax3.bar(['Gripper'], [gripper])
    ax3.set_title('Gripper State')
    ax3.set_ylabel('Value')
    ax3.grid(True, alpha=0.3)
    
    # 4. EEF Position
    ax4 = fig.add_subplot(gs[1, 0])
    eef_position = eef_data[:3]
    ax4.bar(['X', 'Y', 'Z'], eef_position)
    ax4.set_title('EEF Position')
    ax4.set_ylabel('Position (m)')
    ax4.grid(True, alpha=0.3)
    
    # 5. EEF Orientation (Quaternion)
    ax5 = fig.add_subplot(gs[1, 1])
    eef_quat = eef_data[3:]  # [qx, qy, qz, qw]
    ax5.bar(['Qx', 'Qy', 'Qz', 'Qw'], eef_quat)
    ax5.set_title('EEF Orientation (Quaternion)')
    ax5.set_ylabel('Quaternion Value')
    ax5.grid(True, alpha=0.3)
    
    # 6. 3D visualization of arm configuration (simple representation)
    ax6 = fig.add_subplot(gs[1, 2], projection='3d')
    
    # Visualize arm joints as a simple chain (cumulative angles)
    arm_positions = np.zeros((len(arm_joints) + 1, 3))
    current_angle = 0
    for i, angle in enumerate(arm_joints):
        current_angle += angle
        arm_positions[i + 1, 0] = np.sum(np.cos(arm_joints[:i+1]))
        arm_positions[i + 1, 1] = np.sum(np.sin(arm_joints[:i+1]))
    
    ax6.plot(arm_positions[:, 0], arm_positions[:, 1], arm_positions[:, 2], 'o-', linewidth=2, markersize=5)
    ax6.set_title('Arm Configuration (2D projection)')
    ax6.set_xlabel('X')
    ax6.set_ylabel('Y')
    ax6.set_zlabel('Z')
    
    # 7. Display all data as text
    ax7 = fig.add_subplot(gs[2, :])
    ax7.axis('off')
    
    info_text = f"""
    Mobile Base: X={mobile_base[0]:.4f}, Y={mobile_base[1]:.4f}, Theta={mobile_base[2]:.4f}
    Arm Joints (rad): {', '.join([f'{v:.4f}' for v in arm_joints])}
    Gripper: {gripper:.4f}
    
    EEF Position (m): X={eef_position[0]:.4f}, Y={eef_position[1]:.4f}, Z={eef_position[2]:.4f}
    EEF Orientation (Quaternion): Qx={eef_quat[0]:.4f}, Qy={eef_quat[1]:.4f}, Qz={eef_quat[2]:.4f}, Qw={eef_quat[3]:.4f}
    """
    
    ax7.text(0.05, 0.5, info_text, transform=ax7.transAxes, 
            fontfamily='monospace', fontsize=10, verticalalignment='center')
    
    plt.tight_layout()
    plt.show()

# Create slider for Joint/EEF visualization
joint_eef_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_timesteps - 1,
    step=1,
    description='Timestep:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(widgets.interactive(visualize_joint_and_eef, step=joint_eef_slider))

interactive(children=(IntSlider(value=0, description='Timestep:', layout=Layout(width='500px'), max=31, style=…